# 04 - PPO vs DPO Comparison

This notebook compares PPO and DPO alignment approaches:
- Understanding PPO and DPO algorithms
- Training both methods on the same data
- Head-to-head comparison
- Trade-offs analysis

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np

from alignment.ppo_trainer import PPOTrainer, PPOConfig
from alignment.dpo_trainer import DPOTrainer, DPOConfig, DPOLossType
from alignment.comparison import AlignmentComparator

## 1. PPO Overview

PPO (Proximal Policy Optimization) is the standard RLHF approach:

1. Generate responses from the policy
2. Score responses with reward model
3. Update policy to maximize reward while staying close to reference

$$L_{PPO} = \mathbb{E}[\min(r_t A_t, \text{clip}(r_t, 1-\epsilon, 1+\epsilon) A_t)] - \beta D_{KL}(\pi || \pi_{ref})$$

In [ ]:
ppo_config = PPOConfig(
    output_dir="../outputs/ppo_demo",
    total_steps=1000,
    rollout_batch_size=64,
    mini_batch_size=8,
    ppo_epochs=4,
    init_kl_coef=0.1,
    target_kl=6.0,
    cliprange=0.2,
    gamma=1.0,
    lam=0.95,
)

print("PPO Configuration:")
print(f"  KL coefficient: {ppo_config.init_kl_coef}")
print(f"  Target KL: {ppo_config.target_kl}")
print(f"  Clip range: {ppo_config.cliprange}")
print(f"  GAE lambda: {ppo_config.lam}")

## 2. DPO Overview

DPO (Direct Preference Optimization) is a simpler alternative:

1. No explicit reward model needed during training
2. Directly optimize on preference pairs
3. Implicit reward: $r(x, y) = \beta \log \frac{\pi(y|x)}{\pi_{ref}(y|x)}$

$$L_{DPO} = -\mathbb{E}[\log \sigma(\beta(\log\frac{\pi(y_w|x)}{\pi_{ref}(y_w|x)} - \log\frac{\pi(y_l|x)}{\pi_{ref}(y_l|x)}))]$$

In [ ]:
dpo_config = DPOConfig(
    output_dir="../outputs/dpo_demo",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    beta=0.1,
    loss_type=DPOLossType.SIGMOID,
)

print("DPO Configuration:")
print(f"  Beta: {dpo_config.beta}")
print(f"  Loss type: {dpo_config.loss_type}")
print(f"  Learning rate: {dpo_config.learning_rate}")

## 3. Trade-offs

| Aspect | PPO | DPO |
|--------|-----|-----|
| Reward Model | Required | Not needed during training |
| Complexity | Higher | Lower |
| Memory | Higher (generation + RM) | Lower |
| Hyperparameters | Many (KL, clip, GAE) | Few (beta, loss type) |
| Stability | Can be unstable | Generally stable |
| Online Learning | Yes (can iterate) | No (offline only) |

In [ ]:
# Visualize conceptual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training dynamics (simulated)
steps = np.arange(0, 1000, 10)
ppo_reward = 0.5 + 0.4 * (1 - np.exp(-steps/200)) + np.random.normal(0, 0.05, len(steps))
dpo_loss = 0.7 - 0.3 * (1 - np.exp(-steps/300)) + np.random.normal(0, 0.02, len(steps))

axes[0].plot(steps, ppo_reward, label='PPO Reward', color='blue')
axes[0].fill_between(steps, ppo_reward - 0.1, ppo_reward + 0.1, alpha=0.3)
axes[0].set_xlabel('Training Steps')
axes[0].set_ylabel('Mean Reward')
axes[0].set_title('PPO: Reward vs Steps')
axes[0].legend()

axes[1].plot(steps, dpo_loss, label='DPO Loss', color='green')
axes[1].fill_between(steps, dpo_loss - 0.05, dpo_loss + 0.05, alpha=0.3)
axes[1].set_xlabel('Training Steps')
axes[1].set_ylabel('DPO Loss')
axes[1].set_title('DPO: Loss vs Steps')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Comparison Framework

In [ ]:
# Compare trained models (implementation required)
# comparator = AlignmentComparator(
#     reward_model=reward_model,
#     tokenizer=tokenizer,
# )
# result = comparator.compare(
#     model_a=ppo_model,
#     model_b=dpo_model,
#     eval_dataset=eval_dataset,
# )

# Simulated comparison results
metrics = {
    'PPO': {'reward_mean': 1.8, 'kl_div': 5.2, 'length': 180, 'win_rate': 0.52},
    'DPO': {'reward_mean': 1.7, 'kl_div': 4.8, 'length': 165, 'win_rate': 0.48},
}

print("Comparison Results (simulated):")
print(f"{'Metric':<20} {'PPO':>10} {'DPO':>10}")
print("-" * 42)
for metric in ['reward_mean', 'kl_div', 'length', 'win_rate']:
    print(f"{metric:<20} {metrics['PPO'][metric]:>10.2f} {metrics['DPO'][metric]:>10.2f}")

## 5. When to Use Which?

**Use PPO when:**
- You need online learning (iterative improvement)
- You have a good reward model
- You need maximum alignment quality

**Use DPO when:**
- You have limited compute
- You want simpler training
- You have good preference data
- You want to avoid reward model training

## 6. Next Steps

- Analyze reward hacking in `05_reward_hacking_analysis.ipynb`
- Run benchmarks (MT-Bench, AlpacaEval) on both models
- Scale experiments using scaling analysis tools